# A-SCLC

Edit the input path and sample parameters, run the calculator, and inspect the plots.
CSV format: `U(V),I(A),T(C)` with three numeric columns in that order (volts, amperes, Celsius).
The model uses the configured scalar temperature in kelvin; measured temperatures are
available separately as `result.T_K`. Raw measurements retain their signs and order.


In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from asclc_workflow import run_calculation, export_results
from asclc_plotting import (plot_measured_jv, plot_calculated_jv,
                            plot_effective_mobility, plot_carrier_densities,
                            plot_carrier_fraction)


## Input and sample parameters

Example: MAPbBr3 S2, dark. Replace these values for your sample.


In [2]:
input_path = project_root / "data/MAPbBr3_S2_dark.csv"

material = dict(
    E_c=-3.36,       # conduction band edge, eV
    E_v=-5.58,       # valence band edge, eV
    eps_r=25.5,      # relative permittivity
    m_eff_h=0.305,   # hole effective mass / electron mass
    m_eff_e=0.32,    # electron effective mass / electron mass
)
device = dict(
    thickness=6.0e-4,    # m
    area=7.7e-6,         # m^2
    temperature=299.0,   # K, scalar model temperature
    voltage_offset=0.0,  # V
)
model = dict(
    mobility=2.7e-3,  # m^2/(V s)
    E_F0=-4.84,      # equilibrium Fermi level, eV
    E_t=-4.82,       # trap position, eV
    N_t=4.7e16,      # m^-3
    T_t=30.0,        # K
    trap_profile="logistic",  # SI S5 (biexponential); or "gaussian" (SI S6)
    voltage_density="reference",
)
model["gamma"] = model["T_t"] / device["temperature"]

# A4 microscopic mobility, m^2/(V s). Replace with your A3 estimate if available.
analysis_mobility = model["mobility"]


## Numerical settings

The example uses the published SI S5 trap DOS (biexponential, named `logistic` here).
The alternative `gaussian` implements SI S6 with sigma = 2*k_B*T_t/e in eV. Both integrate to N_t.
Other configured conventions are prescribed
model gamma = T_t/T, and voltage density = injected total minus equilibrium free density.
These choices are calculated in Python; their physical limitations are discussed in the guide.
The energy integration excludes its stop; the Fermi sweep includes both endpoints.


In [3]:
numerics = dict(
    energy_range=(0.0, -9.0),  # eV
    fermi_range=(-9.0, 0.0),   # eV
    energy_step=0.003,        # eV
    ohmic_scale=2.0,          # m=1 guide multiplier
    quadratic_scale=2.5,      # m=2 guide multiplier
)


In [4]:
result = run_calculation(input_path, material=material, device=device,
                         model=model, numerics=numerics,
                         analysis_mobility=analysis_mobility)
print(f"Loaded {result.V.size} measurements; "
      f"{(~result.measurements.finite).sum()} nonfinite rows retained.")


Loaded 214 measurements; 0 nonfinite rows retained.


In [ ]:
mu0_estimate = None  # Optional A3 microscopic-mobility estimate, m^2/(V s).
figures = {}          # Collected below and written out by the export cell.


## Measured current density

Measured current density against voltage on logarithmic axes. Nonpositive
readings are kept in the data but cannot appear on a logarithmic axis.
Compare with **Data-calculations, Graf 5** in the workbook.

In [ ]:
fig, ax = plot_measured_jv(result)
figures["measured_jv"] = fig


## Calculated J/V curve

Measured current `j` with the modelled branches `jm (pf)` and `-jm (nf)` and the
`m = 1` (ohmic) and `m = 2` (Mott-Gurney) slope guides. The axes stay on the
measured range. Compare with **MODEL, Graf 2** in the workbook.

In [ ]:
fig, ax = plot_calculated_jv(result)
figures["calculated_jv"] = fig


## A3: Effective mobility

Measured drift mobility `md` (Eq. 4 on adjacent measured intervals), the modelled
`mdm (p)` curve `mu0*|p_f/delta_p|` over the measured voltage range, and the
`mu0 =` reference line. Voltage is linear, mobility logarithmic. The reference
defaults to the model `mobility`; set `mu0_estimate` above to override the line
without changing the model. Compare with **MODEL, Graf 1** in the workbook.

In [ ]:
fig, ax = plot_effective_mobility(result, mu0_estimate=mu0_estimate)
figures["effective_mobility"] = fig


## A4: Free and trapped carrier densities

Measured free `pf` (Eq. 5), trapped `pt` (Eq. 6), and total `ps = pf + pt`,
using `analysis_mobility` from the input cell. Compare with **MODEL, Graf 6** in
the workbook.

In [ ]:
fig, ax = plot_carrier_densities(result)
figures["carrier_densities"] = fig


## A4: Free-carrier fraction

Measured free-carrier fraction Theta = pf / (pf + pt). Compare with
**MODEL, Graf 7** in the workbook.

In [ ]:
fig, ax = plot_carrier_fraction(result)
figures["carrier_fraction"] = fig


## Export

Save every collected figure as PNG/PDF/SVG and write the measured and calculated
CSVs to the output directory.

In [ ]:
output_dir = export_results(result, figures, project_root / "outputs")
print(f"Saved {len(figures)} figures and calculated data to {output_dir}")
